# Benchmarking hadal_flow

In [ ]:
import hadal_flow
import tensorflow as tf
import timeit

context = hadal_flow.create_context64(
    log_n=10,
    main_moduli=[8556589057, 8388812801],
    plaintext_modulus=40961,
    scaling_factor=3,
    seed="test_seed",
)

secret_key = hadal_flow.create_key64(context)
rotation_key = hadal_flow.create_rotation_key64(context, secret_key)

a = tf.random.uniform([context.num_slots, 55555], dtype=tf.float32, maxval=10)
b = tf.random.uniform([55555, 333], dtype=tf.float32, maxval=10)
c = tf.random.uniform([2, context.num_slots], dtype=tf.float32, maxval=10)
d = tf.random.uniform([context.num_slots, 4444], dtype=tf.float32, maxval=10)

enc_a = hadal_flow.to_encrypted(a, secret_key, context)

In [ ]:
def to_pt():
    return hadal_flow.to_shell_plaintext(a, context)

time = min(timeit.Timer(to_pt).repeat(repeat=3, number=1))
print(time)

In [ ]:
def enc():
    return hadal_flow.to_encrypted(d, secret_key, context)

time = min(timeit.Timer(enc).repeat(repeat=3, number=1))
print(time)

In [ ]:
def dec():
    return hadal_flow.to_tensorflow(enc_a, secret_key)

time = min(timeit.Timer(dec).repeat(repeat=3, number=1))
print(time)

In [5]:
def ct_ct_add():
    return enc_a + enc_a

time = min(timeit.Timer(ct_ct_add).repeat(repeat=3, number=1))
print(time)

0.18785935000050813


In [6]:
def ct_ct_sub():
    return enc_a - enc_a

time = min(timeit.Timer(ct_ct_sub).repeat(repeat=3, number=1))
print(time)

0.19354859399754787


In [7]:
def ct_ct_mul():
    return enc_a * 4

time = min(timeit.Timer(ct_ct_mul).repeat(repeat=3, number=1))
print(time)

0.17636438800036558


In [ ]:
def ct_pt_add():
    return enc_a + a

time = min(timeit.Timer(ct_pt_add).repeat(repeat=3, number=1))
print(time)

In [ ]:
def ct_pt_mul():
    return enc_a * a

time = min(timeit.Timer(ct_pt_mul).repeat(repeat=3, number=1))
print(time)

In [ ]:
def ct_pt_matmul():
    return hadal_flow.matmul(enc_a, b)

time = min(timeit.Timer(ct_pt_matmul).repeat(repeat=3, number=1))
print(time)

In [ ]:
def pt_ct_matmul():
    return hadal_flow.matmul(c, enc_a, rotation_key)

time = min(timeit.Timer(pt_ct_matmul).repeat(repeat=3, number=1))
print(time)

In [ ]:
def ct_roll():
    return hadal_flow.roll(enc_a, 2, rotation_key)

time = min(timeit.Timer(ct_roll).repeat(repeat=3, number=1))
print(time)